# <span style="color: Coral;"> Scenario config UI </span>

- Load the <span style="color: orange;"> _utils_ </span> module, which handles the backend functions to provide the <span style="color: blue;"> __scenario comnfig ui__ </span> 

In [ ]:
import bcnexus.utils as utils
import pandas as pd
import re # For regex operations
scenarios_dd, storage_algo_dd, h_grouping_dd, n_clusters_dd, timeslices_label = utils.build_scenario_ui()

- Assign the <span style="color: orange;"> __values__</span>  from <span style="color: blue;"> __scenario UI dropdowns__</span> to <span style="color: orange;"> __arguments__</span> 

In [ ]:
model_builder_args = dict(
    run_scenario=scenarios_dd.value ,
    storage_algorithm=storage_algo_dd.value,
    clustering_attributes=dict(
        hour_grouping=h_grouping_dd.value,
        n_clusters=n_clusters_dd.value
    )
)


# We use the Timeslices information to access run specific folders
timeslices_int = int(re.search(r'\d+', timeslices_label.value).group()) if re.search(r'\d+', timeslices_label.value) else None


### NE_study

In [ ]:
from pathlib import Path
NE_study_root=Path('data/clews_data/NE_study')
NE_LP_file=NE_study_root/'Base_CNZ_NE_FE.lp'
NE_study_csvs=NE_study_root/'storage_case_input_csvs_NE_FE'
NE_results_root=utils.ensure_path('results/NE_FE')
NE_input_updates=NE_study_root/'input_updates'

# <span style="color: Coral;"> Model Runner Object </span>

* <span style="color: grey;">From `clews.runner` module, load the `RunModel` Object </span>  

In [ ]:
from bcnexus.clews.runner import RunModel

* <span style="color: grey;"> Create an instance of the `RunModel` Object `bcnexusRun` with __model_builder_args__ </span>  

In [ ]:
bcnexusRun = RunModel(**model_builder_args)

### 🚀 `run()` — Core Execution Workflow

* <span style="color: grey;"> Apply the `run()` method from  `bcnexusRun` instance </span> to run the complete workflow

About __Arguments__ of the `run()` method:

- `build=True` argument handles the workflow run to build SETs, Ratios, template params etc.
- `include_livestock` handles the livestock modelling workflow using `bcnexus/clews/livestock.py` module.
- `thread` depends on the hardware limitations of your machine. If you have 4 core CPU, use Thread __<=4__ )

* <span style="color: skyblue;"> Where do I get `threads` number ?</span>  
  - <span style="color: grey;"> Apply the `check_machine_cores()` function from  `utils` module. it returns the __number of cores available__ </span>  

In [ ]:
n_physical_cores, n_logical_cores=utils.check_machine_cores()

| Model Characteristics                         | Recommended Threads | Reasoning                                                      |
| --------------------------------------------- | ------------------- | -------------------------------------------------------------- |
| Small LP/MILP (<50k vars)                     | 1–4                 | Over-parallelization adds overhead; single-thread often enough |
| Medium MILP (50k–500k vars)                   | 4–8                 | Good balance of parallel speed-up and stability                |
| Large/complex MILP (>500k vars, branch-heavy) | 8–16                | Parallel branch-and-bound scales moderately well               |
| Nonlinear / Quadratic / MIQP                  | 4–8                 | Solvers often rely on sequential matrix operations             |
| Highly degenerate or network-based LP         | 1–4                 | Parallel simplex often unstable; dual simplex preferred        |


* <span style="color: orange;">  Now run the Model ! </span>  

 <span style="color: coral;"> Important Notes:
  - While creating an instance of `RunModel` object (here named as `bcnexusRun`), it also creates an instance of `BuildModel` object (initiated and named as `ClewsBuilder` inside `RunModel` object ). You can access it via `bcnexusRun.ClewsBuilder`  , and access the methods inside `ClewsBuilder` object. 
  - The `ClewsBuilder` object can be found at </span> <span style="color: biege;"> bcnexus/clews/__builder.py__ module
  - when we set `build=True`. We use the `build()` method of `ClewsBuilder` object.
    - The `build()` method is useful to test/validate new structural changes to the model. This method handles the SETs creation, collection of LandCluster data, technology updates via clews_builder.yaml config. Refer to the script and the example workbook for more details on this object and methods.
    - `ClewsBuilder` has an explicit method `build_SETs_and_ratios` which is an adaptation of the former __clewsy__ tool. Developers can use this method and add their custom methods to incorporate new structural changes to the model.

<span style="color: coral;"> Data flow in folder:</span>
  - Everything happens inside : <span style="color: coral;"> data/clews_data</span>
  - The workflow __creates this directory__ : <span style="color: coral;">data/clews_data/<span style="color: magenta;">clews_build_data </span><br>
      <span style="color: grey;"> It has 3 sub-folders: 
      <span style="color: coral;">
      - input_csvs  <span style="color: grey;">: The workflow collects templates data first, then updates the files accordingly to reflect changes in clews_builder.yaml config. <span style="color: red;"> if `build`=False, then the workflow assumes all files are updated in this folder. Once you build the model structure, to adjust the temporal clustering attributes only, you can set `build=False`, to skip the steps to build this folder.
      - inputs_csv_8760<span style="color: grey;">: Creates BCNexus compatible profiling parameters in hourly resolution. Which are used to apply the clustering. <span style="color: red;"> if `build`=False, then the workflow assumes all files are updated in this folder.Once you build the model structure, to adjust the temporal clustering attributes only, you can set `build=False`, to skip the steps to build this folder.
      - Model_storage_algorithm<span style="color: grey;">: Final datafiles are stored inside this folder with storage case, and sub-folders with scenarios. 
  - Once you build the model structure, yt adjust the clustering attributes only, you can set `build=False`, to skip the steps to build this folder.


In [ ]:
# bcnexusRun.run(build=False,
#              include_livestock=False,
#              threads=32)

In [ ]:
bcnexusRun.clewsBuilder.update_temporal_profiles()
utils.copy_csv_files(src_folder=bcnexusRun.clewsBuilder.clews_build_input_csv_dir,
                                dest_folder=bcnexusRun.clewsBuilder.storage_case_input_csvs,
                                all_files=True)


In [ ]:
utils.copy_csv_files(src_folder=bcnexusRun.clewsBuilder.storage_case_input_csvs,
                    dest_folder=NE_study_csvs,
                    all_files=True)


In [ ]:
bcnexusRun.clewsBuilder.update_storage_case_temporal_schema()

In [ ]:
bcnexusRun.process_scenario_data()

bcnexusRun.timeslices=len(pd.read_csv(bcnexusRun.input_csvs/'TIMESLICE.csv')) # We need this for directories

In [ ]:
utils.copy_csv_files(src_folder=bcnexusRun.clewsBuilder.storage_case_input_csvs,
                    dest_folder=NE_study_csvs,
                    all_files=True)

utils.copy_csv_files(src_folder=NE_input_updates,
                    dest_folder=NE_study_csvs,
                    all_files=True)

In [ ]:
# data_path,model_path=bcnexusRun.get_model_run_files()

_data_file_=bcnexusRun.get_input_datafile_from_csvs(NE_study_csvs)

data_file, model_file = bcnexusRun.preprocess_data_model( data_infile=_data_file_,
                                                model_file=bcnexusRun.org_model_file)


In [ ]:
bcnexusRun.LP_file=NE_LP_file
bcnexusRun.write_LP_file(model_file,
                       data_file,
                       bcnexusRun.LP_file
                       )

In [ ]:
solved_m=bcnexusRun.solve_model_gurobi(bcnexusRun.LP_file,
                            NE_study_root/"gurobi_log.txt")

In [ ]:
bcnexusRun.solution_path = NE_results_root/f'{bcnexusRun.timeslices}ts'/f'{bcnexusRun.timeslices}ts_solution_gurobi.sol' 

In [ ]:
bcnexusRun.write_solution(solved_m,
            bcnexusRun.solution_path)

In [ ]:
# shadow_price_ELCB02=bcnexusRun.get_shadow_price_ELCB02(solved_m,
#                 plot_save_to=NE_results_root/f'{bcnexusRun.timeslices}ts'/f'shadowprice_ELC02_{bcnexusRun.timeslices}ts.png',
#                 show=False)

In [ ]:
bcnexusRun.get_result_csvs(solution_file=bcnexusRun.solution_path,
                           results_save_to=NE_results_root/f'{bcnexusRun.timeslices}ts'/f'{bcnexusRun.timeslices}ts_csvs_gurobi',
                           debug_mode=True)

# <span style="color: Coral;"> Results </span>

### <span style="color: grey;"> Check Results @ `results\clews\<Model_<storage_algorithm>_<scenario>`</span>

- <span style="color: grey;"> We can also use the `datapackage` module's `GetDataPackage` object to load load the results as an object.
- <span style="color: grey;"> This results pack could be used for __post-analysis, result exchange to other models, visualizations, scenario dashes__ etc.

- <span style="color: grey;">  Get `result_pack` as an instance of `GetDataPackage` object

In [ ]:
from bcnexus.clews.datapackage import GetDataPackage
NE_result_csvs=NE_results_root/f'{timeslices_int}ts'/f'{timeslices_int}ts_csvs_gurobi'
result_pack=GetDataPackage(NE_result_csvs)
result_pack.show

- <span style="color: grey;">  Apply `get_dataframe(<Result param name>)` method of the `result_pack` instance

In [ ]:
#example
result_pack.get_dataframe('NewCapacity')

# <span style="color: yellow;"> Plots </span>

!! <span style="color: biege;"> Still under active development to include more comprehensive plots <span style="color: yellow;"> EL_20251119

* <span style="color: grey;">Load the `plotter` module and use the `main()` function </span>

In [ ]:
from pathlib import Path
import bcnexus.plots as plotter
NE_vis_root=utils.ensure_path(Path(f'vis/NE_FE/{model_builder_args["run_scenario"]}/{timeslices_int}ts'))
plotter.main(nexus_scenario=model_builder_args['run_scenario'],
             storage_algorithm=model_builder_args['storage_algorithm'],
             timeslices=timeslices_int,
             results_csvs=NE_result_csvs,
             plots_save_to=NE_vis_root)

- <span style="color: biege;">  Load the `plotter` module. 
  - <span style="color: grey;"> plotter.`get_plots()` function returns a dictionary of plots. You can review the plots in this notebook from that dictionary. The function also save the plots as html to 'vis/bccm/< scenario > '

In [ ]:
NE_result_csvs

In [ ]:
import bcnexus.plots as plotter

plotter_args=dict(
    nexus_scenario=model_builder_args['run_scenario'],
    storage_algorithm=model_builder_args['storage_algorithm'],
    timeslices=timeslices_int,
    results_csvs=NE_result_csvs,
    plots_save_to=NE_vis_root
)
# plotter.main(**plotter_args)
nexus_plots=plotter.get_plots(**plotter_args)

In [ ]:
for type,plot_dicts in nexus_plots.items():
    utils.print_update(level=1,message=f"Type: {type}")
    for plot_name,plot in plot_dicts.items():
        utils.print_update(level=2,message=f"Plot name: {plot_name}")

In [ ]:
nexus_plots['Climate']['emission_total']

In [ ]:
nexus_plots['Energy']['capacity_investments']